In [ ]:
import sys
print(sys.executable)


In [ ]:
!"C:\Users\priyansh\torch-intel\Scripts\python.exe" -m pip install optuna


In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [ ]:
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
device

In [ ]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

In [ ]:
class_names = dataset.classes
class_names 

In [ ]:
num_classes = len(dataset.classes)
num_classes

In [ ]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

In [ ]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

### Training & Hyperparameter tuning 

In [ ]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Subset, DataLoader

# ----------------------------
# 1) Make a SMALL train loader for Optuna tuning (FAST)
# ----------------------------
subset_size = min(3000, len(train_loader.dataset))  # change 3000 -> 2000/5000 if needed
indices = np.random.choice(len(train_loader.dataset), subset_size, replace=False)

train_subset = Subset(train_loader.dataset, indices)

train_loader_small = DataLoader(
    train_subset,
    batch_size=32,        # keep small for laptop
    shuffle=True,
    num_workers=0,        # best for Windows stability
    pin_memory=False
)

# ----------------------------
# 2) Objective function (FAST tuning)
# ----------------------------
def objective(trial):
    # Hyperparameters
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)  # smaller realistic range

    # Model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)

    # Loss + Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    epochs = 1  # IMPORTANT: keep 1 for tuning speed (use 2 max)

    for epoch in range(epochs):
        # ---- Train ----
        model.train()
        for images, labels in train_loader_small:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # ---- Validate ----
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100.0 * correct / total

        # Report to Optuna
        trial.report(accuracy, epoch)

        # Prune bad trials early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

# ----------------------------
# 3) Study with PRUNING (FAST)
# ----------------------------
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=0)

study = optuna.create_study(direction="maximize", pruner=pruner)

# Keep trials low for laptop
study.optimize(objective, n_trials=15)

print("Best Trial Accuracy:", study.best_value)
print("Best Params:", study.best_params)


In [ ]:
study.best_params

#### the best learning rate is 0.004 and dropout rate is 0.2